In [129]:
import numpy as np
import pandas as pd
import os
import sys
import seaborn as sns
import pickle
import matplotlib.pyplot as plt
import scipy.stats as stats 
import missingno as msno
import sklearn as sk
import optuna
from lightgbm import LGBMRegressor
from sklearn.impute import KNNImputer
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, StandardScaler, PowerTransformer
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from feature_engine.outliers import Winsorizer
sys.path.append("../src/Preprocesamiento")
from custom_transformers import OutlierHandler
import matplotlib.pyplot as plt
import plotly

In [88]:
SEMILLA = 42
FILENAME = "dataset_final.csv"
datapath = os.path.join("..","data","processed",FILENAME)
df = pd.read_csv(datapath)

In [89]:
y = df['target_Frio_kW_next']
df = df.drop(columns=['target_Frio_kW_next'])

In [90]:
with open ("../src/Preprocesamiento/preprocesamiento.pkl", "rb") as f:
    preprocessor = pickle.load(f)

In [91]:
df_preprocesado = preprocessor.transform(df)


### Separación de datos

In [92]:
X_train, X_test, y_train, y_test = train_test_split(df_preprocesado, y, train_size=0.7, shuffle=False, random_state=SEMILLA)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, train_size=0.5, shuffle=False, random_state=SEMILLA)


### Baselines: MAE

Día de hoy

In [93]:
df_sin_escalar = preprocessor.named_steps['Escalado'].set_output(transform="pandas").inverse_transform(X_val)
df_sin_escalar = pd.DataFrame(df_sin_escalar, columns=df.columns)

In [94]:
y_val

981     22816.0
982     23396.0
983     22853.0
984     28453.0
985     18731.0
         ...   
1150     9481.0
1151    11587.0
1152    10123.0
1153    10360.0
1154    10568.0
Name: target_Frio_kW_next, Length: 174, dtype: float64

In [95]:
y_frio_hoy = df_sin_escalar['Frio (Kw)'].values
MAE_frio_hoy = np.mean(np.abs(y_frio_hoy - y_val))
print(f"MAE entre Frio (Kw) y target_Frio_kW_next: {MAE_frio_hoy}")

MAE entre Frio (Kw) y target_Frio_kW_next: 3476.6666666666665


Promedio móvil

In [96]:
y_prom_movil = df_sin_escalar['Prom movil Frio'].values
MAE_prom_movil = np.mean(np.abs(y_prom_movil - y_val))
print(f"MAE entre Prom movil Frio y target_Frio_kW_next: {MAE_prom_movil:.4f}")

MAE entre Prom movil Frio y target_Frio_kW_next: 3058.8415


## Entrenamiento de modelos

### GXBoost

In [105]:
gxboost = GradientBoostingRegressor(random_state=SEMILLA)
gxboost.fit(X_train, y_train)
y_pred = gxboost.predict(X_val)
MAE_gxboost = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_gxboost:.4f}")

MAE: 3729.3547


### Random Forest Regressor

In [106]:
randomforest = RandomForestRegressor(random_state=SEMILLA)
randomforest.fit(X_train, y_train)
MAE_randomforest = np.mean(np.abs(randomforest.predict(X_val) - y_val))
print(f"MAE: {MAE_randomforest:.4f}")

MAE: 3169.0072


### lightGBM

In [107]:
lightgbm = LGBMRegressor(random_state=SEMILLA)
lightgbm.fit(X_train, y_train)
y_pred = lightgbm.predict(X_val)
MAE_lightgbm = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_lightgbm:.4f}")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000327 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3575
[LightGBM] [Info] Number of data points in the train set: 808, number of used features: 16
[LightGBM] [Info] Start training from score 24985.313119
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positiv

### Lasso

In [108]:
lasso = Lasso(random_state=SEMILLA)
lasso.fit(X_train, y_train)
y_pred = lasso.predict(X_val)
MAE_lasso = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_lasso:.4f}")

MAE: 3233.8368


In [ ]:

# log_file = "results/experiments_log.csv"
# exp_id = "exp_01"
# if os.path.exists(log_file):
#     log_df = pd.read_csv(log_file,sep=',')
#     if not log_df.empty:
#         exp_id = log_df.iloc[-1]['experiment_id']
#         exp_id = int(exp_id.split('_')[-1]) + 1
#         exp_id = f"exp_{exp_id:02d}" 

# log_data = pd.DataFrame([{
#     "experiment_id": exp_id,
#     "datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#     "model_version": "v1",
#     "hyperparameters": "RandomForest(n_estimators=100, random_state=42)",
#     "metrics": f"MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.2f}"
# }])

# if not os.path.exists(log_file):
#     log_data.to_csv(log_file, index=False,header=True)
# else:
#     log_data.to_csv(log_file, mode='a', header=False, index=False)

### Busqueda de hiperparámetros

### GXBoost

In [121]:
def objective(trial):
   n_estimators = trial.suggest_int("n_estimators", 50,200)
   max_depth = trial.suggest_int("max_depth", 3, 20)
   min_samples_split = trial.suggest_int("min_samples_split", 2,10)
   learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
   alpha = trial.suggest_float("alpha", 0.0, 0.1)
   
   model = GradientBoostingRegressor(
       n_estimators= n_estimators,
       max_depth= max_depth,
       min_samples_split= min_samples_split,
       learning_rate= learning_rate,
       alpha = alpha,
       random_state=SEMILLA
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score

study = optuna.create_study(direction = "minimize")
study.optimize(objective, n_trials=50)

[I 2026-11-13 00:27:19,843] A new study created in memory with name: no-name-d4520c9a-00a2-41f1-83e4-b5e8782720c0
[I 2026-11-13 00:27:20,714] Trial 0 finished with value: 3736.954305032433 and parameters: {'n_estimators': 128, 'max_depth': 20, 'min_samples_split': 5, 'learning_rate': 0.08655112605513683, 'alpha': 0.02054784437770848}. Best is trial 0 with value: 3736.954305032433.
[I 2026-11-13 00:27:21,790] Trial 1 finished with value: 3998.3770739917186 and parameters: {'n_estimators': 169, 'max_depth': 18, 'min_samples_split': 4, 'learning_rate': 0.1122395826144233, 'alpha': 0.04363591689718184}. Best is trial 0 with value: 3736.954305032433.
[I 2026-11-13 00:27:22,289] Trial 2 finished with value: 3632.646854979956 and parameters: {'n_estimators': 104, 'max_depth': 9, 'min_samples_split': 5, 'learning_rate': 0.05714279914286603, 'alpha': 0.024140644410555724}. Best is trial 2 with value: 3632.646854979956.
[I 2026-11-13 00:27:23,178] Trial 3 finished with value: 3732.9532966786714 

In [131]:
import optuna.visualization as vis
import plotly

In [132]:

fig_history = vis.plot_optimization_history(study)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study)
fig_history.show()
fig_contour = vis.plot_contour(study)
fig_contour.show()


fig_importance = vis.plot_param_importances(study)
fig_importance.show()

ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

### RandomForest

In [ ]:
def objective(trial):
   n_estimators = trial.suggest_int("n_estimators", 50,200)
   max_depth = trial.suggest_int("max_depth", 3, 20)
   min_samples_split = trial.suggest_int("min_samples_split", 2,10)
   
   
   model = RandomForestRegressor(
       n_estimators= n_estimators,
       max_depth= max_depth,
       min_samples_split= min_samples_split,
       random_state=SEMILLA
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score

study = optuna.create_study(direction = "minimize")
study.optimize(objective, n_trials=50)

[I 2026-11-13 00:15:52,006] A new study created in memory with name: no-name-d614e580-109f-46ba-a66b-c2dd7e12f164
[I 2026-11-13 00:15:52,218] Trial 0 finished with value: 3056.6813417146514 and parameters: {'n_estimators': 111, 'max_depth': 3, 'min_samples_split': 6}. Best is trial 0 with value: 3056.6813417146514.
[I 2026-11-13 00:15:52,960] Trial 1 finished with value: 3131.934785148525 and parameters: {'n_estimators': 172, 'max_depth': 12, 'min_samples_split': 5}. Best is trial 0 with value: 3056.6813417146514.
[I 2026-11-13 00:15:53,346] Trial 2 finished with value: 3151.35518711105 and parameters: {'n_estimators': 88, 'max_depth': 19, 'min_samples_split': 5}. Best is trial 0 with value: 3056.6813417146514.
[I 2026-11-13 00:15:53,679] Trial 3 finished with value: 3180.222747690848 and parameters: {'n_estimators': 88, 'max_depth': 11, 'min_samples_split': 4}. Best is trial 0 with value: 3056.6813417146514.
[I 2026-11-13 00:15:53,925] Trial 4 finished with value: 3212.180828716135 an

### lightGBM

In [119]:
def objective(trial):
   n_estimators = trial.suggest_int("n_estimators", 50,200)
   max_depth = trial.suggest_int("max_depth", 3, 20)
   learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
   reg_alpha = trial.suggest_float("reg_alpha", 0.0, 0.1)
   reg_lambda = trial.suggest_float("reg_lambda", 0.0, 0.1)
   
   model = LGBMRegressor(
       n_estimators= n_estimators,
       max_depth= max_depth,
       learning_rate= learning_rate,
       reg_alpha= reg_alpha,
       reg_lambda= reg_lambda,
       random_state=SEMILLA,
       verbose=-1
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score

study = optuna.create_study(direction = "minimize")
study.optimize(objective, n_trials=50)

[I 2026-11-13 00:23:35,710] A new study created in memory with name: no-name-f78db893-69cc-4b85-9568-b8f5ceeedbdf
[I 2026-11-13 00:23:35,879] Trial 0 finished with value: 3578.3827357968908 and parameters: {'n_estimators': 178, 'max_depth': 7, 'learning_rate': 0.27948061445269773, 'reg_alpha': 0.053229341636211305, 'reg_lambda': 0.05150439525001396}. Best is trial 0 with value: 3578.3827357968908.
[I 2026-11-13 00:23:35,992] Trial 1 finished with value: 3559.322540984401 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.11252992390171991, 'reg_alpha': 0.0841519696311343, 'reg_lambda': 0.05240109946854629}. Best is trial 1 with value: 3559.322540984401.
[I 2026-11-13 00:23:36,014] Trial 2 finished with value: 3313.953419749912 and parameters: {'n_estimators': 70, 'max_depth': 4, 'learning_rate': 0.08640679534723002, 'reg_alpha': 0.06185386742617224, 'reg_lambda': 0.07826047848637413}. Best is trial 2 with value: 3313.953419749912.
[I 2026-11-13 00:23:36,150] Trial

### Lasso

In [120]:
def objective(trial):
   alpha = trial.suggest_float("alpha", 0.0, 0.1)
   
   model = Lasso(
       alpha= alpha,
       random_state=SEMILLA,
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   score = np.mean(np.abs(y_pred - y_val))
   return score

study = optuna.create_study(direction = "minimize")
study.optimize(objective, n_trials=50)

[I 2026-11-13 00:26:36,908] A new study created in memory with name: no-name-c178c437-50cc-473f-bf4f-3bbe70e81048
[I 2026-11-13 00:26:36,932] Trial 0 finished with value: 3239.940465359543 and parameters: {'alpha': 0.01922428006388606}. Best is trial 0 with value: 3239.940465359543.
[I 2026-11-13 00:26:36,944] Trial 1 finished with value: 3239.7014358220463 and parameters: {'alpha': 0.05603829402084787}. Best is trial 1 with value: 3239.7014358220463.
[I 2026-11-13 00:26:36,960] Trial 2 finished with value: 3239.5308285875235 and parameters: {'alpha': 0.08230868077800695}. Best is trial 2 with value: 3239.5308285875235.
[I 2026-11-13 00:26:36,972] Trial 3 finished with value: 3239.5102055605403 and parameters: {'alpha': 0.08549329540327338}. Best is trial 3 with value: 3239.5102055605403.
[I 2026-11-13 00:26:36,985] Trial 4 finished with value: 3239.752892087888 and parameters: {'alpha': 0.04810998371611702}. Best is trial 3 with value: 3239.5102055605403.
[I 2026-11-13 00:26:37,005] T